# Data Collection and Pre-Processing Lab

This notebook demonstrates an end-to-end data engineering workflow using an e-commerce sales dataset.

The workflow includes data ingestion, data structures, profiling, cleaning, transformation, feature engineering, aggregation, serialization, and reflection.

## Step 1 — Hello, Data!

The first step is to load the raw e-commerce CSV file without modifying the source data. The first three rows are displayed to verify that the dataset was loaded correctly.

In [25]:
from pathlib import Path
import sys
import pandas as pd

# Find the project root
PROJECT_ROOT = Path.cwd()

# If the notebook is running from the notebooks folder,
# move one level up to the project root.
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Add the project root to Python's import path
sys.path.append(str(PROJECT_ROOT))

# Define important folders
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "output"

# Primary dataset path
RAW_DATA_PATH = DATA_DIR / "retail_sales_ontario_synthetic.csv"

# Load the raw CSV
sales_data = pd.read_csv(
    RAW_DATA_PATH,
    low_memory=False
)

# Display the first 3 rows
sales_data.head(3)

,order_id,date,customer_id,product_id,product,product_category,price,quantity,coupon_code,discount_pct,payment_method,shipping_city,shipping_province,sales_amount
0,ON100762,03/01/2025,C11886,P2378,Lip Balm,Beauty,63.41,6.0,NO_COUPON,0,Credit Card,London,ON,380.46
1,ON100888,03/01/2025,C10858,P1339,Air Fryer,Home & Kitchen,55.39,5.0,WELCOME5,5,Interac,Guelph,ON,263.10
2,ON100375,04/01/2025,C12057,P8749,Coffee Maker,Home & Kitchen,81.91,3.0,NO_COUPON,0,Credit Card,Hamilton,ON,245.73


In [26]:
sales_data.columns.tolist()

['order_id',
 'date',
 'customer_id',
 'product_id',
 'product',
 'product_category',
 'price',
 'quantity',
 'coupon_code',
 'discount_pct',
 'payment_method',
 'shipping_city',
 'shipping_province',
 'sales_amount']

## Step 2 — Pick the Right Container

A dictionary is appropriate for a sales record because it stores related values using meaningful field names such as `customer_id`, `product`, and `price`, and it is easy to update. A set is useful when we only need unique values, such as unique shipping cities, while a namedtuple would be better for fixed, immutable records.

## Step 3 — Implement Functions and Data Structure

The `LoadSales` class in `src/Load_data.py` uses object-oriented programming to load and clean the sales data. The loaded DataFrame is also converted into a list of dictionaries so that each transaction is represented as a Python data structure.

In [27]:
from src.Load_data import LoadSales

# Create an object from our class
loader = LoadSales(RAW_DATA_PATH)

# Load the sales data
sales_data = loader.getSales()

# Convert the DataFrame into a list of dictionaries
sales_records = sales_data.to_dict(orient="records")

# Check the resulting data structure
print("Data structure type:", type(sales_records))
print("Number of records:", len(sales_records))

# Display the first record
sales_records[0]

Data structure type: <class 'list'>
Number of records: 1020


{'order_id': 'ON100762',
 'date': '03/01/2025',
 'customer_id': 'C11886',
 'product_id': 'P2378',
 'product': 'Lip Balm',
 'product_category': 'Beauty',
 'price': 63.41,
 'quantity': 6.0,
 'coupon_code': 'NO_COUPON',
 'discount_pct': 0,
 'payment_method': 'Credit Card',
 'shipping_city': 'London',
 'shipping_province': 'ON',
 'sales_amount': 380.46}

## Step 4 — Bulk Loaded

A DataFrame is useful for bulk data processing, while dictionaries are useful for lookup operations. The following example creates a dictionary mapping each customer ID to a shipping city.

In [28]:
customer_city_map = (
    sales_data[
        ["customer_id", "shipping_city"]
    ]
    .dropna()
    .drop_duplicates("customer_id")
    .set_index("customer_id")["shipping_city"]
    .to_dict()
)

# Display a few dictionary entries
list(customer_city_map.items())[:5]

[('C11886', 'London'),
 ('C10858', 'Guelph'),
 ('C12057', 'Hamilton'),
 ('C10823', 'Vaughan'),
 ('C10951', 'Scarborough')]

## Step 5 — Quick Profiling

Before cleaning the dataset, we need a quick profile. This includes the minimum, mean, and maximum product price and the number of unique shipping cities. A Python `set` is used to identify unique cities.

In [29]:
# Make sure price is numeric for profiling
sales_data["price"] = pd.to_numeric(
    sales_data["price"],
    errors="coerce"
)

# Price statistics
minimum_price = sales_data["price"].min()
average_price = sales_data["price"].mean()
maximum_price = sales_data["price"].max()

# Use a set to count unique cities
unique_cities = set(
    sales_data["shipping_city"]
    .dropna()
    .astype(str)
    .str.strip()
)

print(f"Minimum price: ${minimum_price:.2f}")
print(f"Mean price: ${average_price:.2f}")
print(f"Maximum price: ${maximum_price:.2f}")
print(f"Unique shipping cities: {len(unique_cities)}")

Minimum price: $3.10
Mean price: $57.68
Maximum price: $257.97
Unique shipping cities: 15


## Step 6 — Spot the Grime

The raw data is checked for common data-quality problems before cleaning. The checks include missing required values, duplicate rows, invalid dates, non-positive prices or quantities, and inconsistent text formatting such as extra whitespace.

In [30]:
# Check for common data-quality problems

invalid_dates = (
    pd.to_datetime(
        sales_data["date"],
        errors="coerce"
    ).isna().sum()
)

missing_required = (
    sales_data[
        [
            "date",
            "customer_id",
            "product",
            "price",
            "quantity",
            "shipping_city"
        ]
    ]
    .isna()
    .any(axis=1)
    .sum()
)

duplicate_rows = sales_data.duplicated().sum()

non_positive_price = (
    pd.to_numeric(
        sales_data["price"],
        errors="coerce"
    ) <= 0
).sum()

non_positive_quantity = (
    pd.to_numeric(
        sales_data["quantity"],
        errors="coerce"
    ) <= 0
).sum()

messy_city_names = (
    sales_data["shipping_city"]
    .dropna()
    .astype(str)
    .apply(lambda x: x != x.strip())
    .sum()
)

dirty_report = {
    "Missing required values": int(missing_required),
    "Invalid dates": int(invalid_dates),
    "Duplicate rows": int(duplicate_rows),
    "Non-positive prices": int(non_positive_price),
    "Non-positive quantities": int(non_positive_quantity),
    "Cities with extra whitespace": int(messy_city_names)
}

pd.Series(
    dirty_report,
    name="Number of affected rows"
)

Missing required values           4
Invalid dates                   635
Duplicate rows                    0
Non-positive prices               0
Non-positive quantities           0
Cities with extra whitespace      0
Name: Number of affected rows, dtype: int64

In [31]:
# Show examples of rows with missing required values
sales_data[
    sales_data[
        [
            "date",
            "customer_id",
            "product",
            "price",
            "quantity",
            "shipping_city"
        ]
    ]
    .isna()
    .any(axis=1)
].head(5)

,order_id,date,customer_id,product_id,product,product_category,price,quantity,coupon_code,discount_pct,payment_method,shipping_city,shipping_province,sales_amount
937,ON100520,15/07/2026,C11491,P7140,T-Shirt,Apparel,NaN,5.0,WELCOME5,5,Debit Card,London,ON,241.44
977,ON100445,06/08/2026,C10901,P2488,Bluetooth Speaker,Electronics,86.45,NaN,SAVE10,10,Debit Card,Waterloo,ON,77.81
1003,ON100587,18/08/2026,C12462,P3140,Bluetooth Speaker,Electronics,21.55,5.0,SAVE10,10,Debit Card,NaN,ON,96.98
1014,ON100504,27/08/2026,C12088,P6690,Granola Bars,Grocery,NaN,2.0,NaN,0,Debit Card,Guelph,ON,14.32


In [32]:
# Show examples of non-positive prices
sales_data[
    pd.to_numeric(
        sales_data["price"],
        errors="coerce"
    ) <= 0
].head(5)

,order_id,date,customer_id,product_id,product,product_category,price,quantity,coupon_code,discount_pct,payment_method,shipping_city,shipping_province,sales_amount


## Step 7 — Cleaning Rules

The cleaning rules are implemented inside the `clean()` method of the `LoadSales` class. The rules standardize text, convert dates and numeric columns, remove rows with missing required fields, remove invalid prices and quantities, fix discount values, and remove exact duplicate rows.

In [33]:
# Record the number of rows before cleaning
rows_before = len(sales_data)

# Execute the cleaning rules inside the class
cleaned_sales = loader.clean()

# Record the number of rows after cleaning
rows_after = len(cleaned_sales)

print(f"Rows before cleaning: {rows_before}")
print(f"Rows after cleaning: {rows_after}")
print(f"Rows removed: {rows_before - rows_after}")

print(
    "Missing values after cleaning:",
    int(cleaned_sales.isna().sum().sum())
)

print(
    "Duplicate rows after cleaning:",
    int(cleaned_sales.duplicated().sum())
)

Rows before cleaning: 1020
Rows after cleaning: 384
Rows removed: 636
Missing values after cleaning: 0
Duplicate rows after cleaning: 0


## Step 8 — Transformations

The `coupon_code` field is transformed into a numeric `coupon_discount_pct` field by extracting the numeric portion of the coupon code. A value of 0 is used when no numeric discount is present. Gross and net revenue are also calculated from the cleaned price, quantity, and discount values.

In [34]:
# Convert the coupon code into a numeric discount percentage
cleaned_sales["coupon_discount_pct"] = (
    cleaned_sales["coupon_code"]
    .astype("string")
    .str.extract(r"(\d+(?:\.\d+)?)")[0]
    .fillna(0)
    .astype(float)
)

# Calculate gross revenue
cleaned_sales["gross_revenue"] = (
    cleaned_sales["price"]
    * cleaned_sales["quantity"]
)

# Calculate net revenue after the existing discount percentage
cleaned_sales["net_revenue"] = (
    cleaned_sales["gross_revenue"]
    * (1 - cleaned_sales["discount_pct"] / 100)
)

cleaned_sales[
    [
        "coupon_code",
        "coupon_discount_pct",
        "price",
        "quantity",
        "gross_revenue",
        "net_revenue"
    ]
].head(10)

,coupon_code,coupon_discount_pct,price,quantity,gross_revenue,net_revenue
0,NO_COUPON,0.0,63.41,6.0,380.46,380.4600
1,WELCOME5,5.0,55.39,5.0,276.95,263.1025
2,NO_COUPON,0.0,81.91,3.0,245.73,245.7300
3,NO_COUPON,0.0,93.35,3.0,280.05,280.0500
4,NO_COUPON,0.0,14.40,3.0,43.20,43.2000
5,NO_COUPON,0.0,9.20,2.0,18.40,18.4000
6,WELCOME5,5.0,206.19,3.0,618.57,587.6415
7,SAVE10,10.0,16.79,2.0,33.58,30.2220
8,FREESHIP,0.0,14.66,1.0,14.66,14.6600
9,NO_COUPON,0.0,153.10,1.0,153.10,153.1000


## Step 9 — Feature Engineering

A new `days_since_purchase` feature is created to show how old each transaction is relative to the latest purchase date in the cleaned dataset. Using the latest dataset date as the reference makes the calculation reproducible.

In [ ]:
# Use the latest purchase date as the reference date
reference_date = cleaned_sales["date"].max().normalize()

# Calculate days since purchase
cleaned_sales["days_since_purchase"] = (
    reference_date
    - cleaned_sales["date"].dt.normalize()
).dt.days

print("Reference date:", reference_date)

cleaned_sales[
    [
        "date",
        "days_since_purchase"
    ]
].head(10)